# RandLA-Net Using Open3D-ML

https://github.com/isl-org/Open3D-ML?tab=readme-ov-file#semantic-segmentation

In [1]:
%load_ext autoreload
%autoreload 2

In [8]:
import os
import open3d.ml as _ml3d
import open3d.ml.torch as ml3d
import ml3d.torch as ml3d


cfg_file = "/home/arthur/Documents/Code/Github/Open3D-ML/ml3d/configs/randlanet_semantickitti.yml"
cfg = _ml3d.utils.Config.load_from_file(cfg_file)

In [9]:
model = ml3d.models.RandLANet(**cfg.model)
cfg.dataset['dataset_path'] = '/media/arthur/HDD/Datasets/Point Clouds/SemanticKitti/raw/'
dataset = _ml3d.datasets.SemanticKITTI(cfg.dataset.pop('dataset_path', None), **cfg.dataset)
pipeline = ml3d.pipelines.SemanticSegmentation(model, dataset=dataset, device="gpu", **cfg.pipeline)

In [10]:
# download the weights.
ckpt_folder = "./logs/"
os.makedirs(ckpt_folder, exist_ok=True)
ckpt_path = ckpt_folder + "randlanet_semantickitti_202201071330utc.pth"
randlanet_url = "https://storage.googleapis.com/open3d-releases/model-zoo/randlanet_semantickitti_202201071330utc.pth"
if not os.path.exists(ckpt_path):
    cmd = "wget {} -O {}".format(randlanet_url, ckpt_path)
    os.system(cmd)

In [11]:
# load the parameters.
pipeline.load_ckpt(ckpt_path=ckpt_path)

In [12]:
test_split = dataset.get_split("test")
data = test_split.get_data(0)

In [13]:
data

{'point': array([[ 1.35929756e+01,  7.97351729e-03,  6.69000268e-01],
        [ 1.35728445e+01,  5.08311652e-02,  6.68001473e-01],
        [ 1.35917816e+01,  7.17616528e-02,  6.69002056e-01],
        ...,
        [ 7.93910265e+00, -2.63877869e+00, -3.77945089e+00],
        [ 7.96492624e+00, -2.61984015e+00, -3.78844428e+00],
        [ 7.98275280e+00, -2.59890866e+00, -3.79343700e+00]], dtype=float32),
 'feat': None,
 'label': array([0, 0, 0, ..., 0, 0, 0], dtype=int32)}

In [14]:
# run inference on a single example.
# returns dict with 'predict_labels' and 'predict_scores'.
result = pipeline.run_inference(data)

test 0/1: 100%|█████████▉| 79701/79845 [00:02<00:00, 32672.65it/s]/home/arthur/Documents/Code/Github/Open3D-ML/ml3d/torch/modules/metrics/semseg_metric.py:54: RuntimeWarning: Mean of empty slice
  accs.append(np.nanmean(accs))
/home/arthur/Documents/Code/Github/Open3D-ML/ml3d/torch/modules/metrics/semseg_metric.py:87: RuntimeWarning: Mean of empty slice
  ious.append(np.nanmean(ious))


In [31]:
import torch
from torch.utils.data import DataLoader

from ml3d.torch.dataloaders import get_sampler, TorchDataloader
from ml3d.datasets import InferenceDummySplit


batcher = pipeline.get_batcher("cpu")
infer_dataset = InferenceDummySplit(data)
pipeline.dataset_split = infer_dataset

infer_sampler = infer_dataset.sampler
infer_split = TorchDataloader(dataset=infer_dataset,
                                preprocess=model.preprocess,
                                transform=model.transform,
                                sampler=infer_sampler,
                                use_cache=False)

infer_loader = DataLoader(infer_split,
                            batch_size=2,
                            sampler=get_sampler(infer_sampler),
                            collate_fn=batcher.collate_fn)

model.trans_point_sampler = infer_sampler.get_point_sampler()

In [48]:
data['point'].shape

(127405, 3)

In [32]:
curr_cloud_id = -1
test_probs = []
ori_test_probs = []
ori_test_labels = []

with torch.no_grad():
    for unused_step, inputs in enumerate(infer_loader):
        results = model(inputs['data'])
        break

In [33]:
len(inputs['data']['coords'])

4

In [34]:
inputs['data']['coords'][3].shape

torch.Size([2, 704, 3])

In [20]:
inputs['data']['neighbor_indices']

[tensor([[[    0, 39185, 11503,  ..., 21495, 25379, 28748],
          [    1, 16100,  4242,  ..., 26698, 32836, 10442],
          [    2,  7485, 22153,  ...,  9404,  7939, 20519],
          ...,
          [45053, 17941, 31114,  ...,  1790, 38028, 14189],
          [45054, 31774, 23866,  ..., 22289, 42448, 12305],
          [45055, 36691,  6487,  ..., 43817, 32965, 18280]]]),
 tensor([[[    0,  2177,  1145,  ..., 10323,  3308,  3018],
          [    1,  4242,  7315,  ...,  8062,  6280,   853],
          [    2,  7485,  3251,  ...,  6379,  8162, 10549],
          ...,
          [11261,  2528,  6311,  ...,  5953,   217,  9202],
          [11262,  5051,  2216,  ...,  3837,  8071,  4969],
          [11263,  7985,  4528,  ...,  7748,   592, 10604]]]),
 tensor([[[   0, 2177, 1145,  ..., 2729,  455,   30],
          [   1, 1247,  506,  ..., 1033, 1347, 2466],
          [   2,  625,  590,  ..., 2585,   69, 1236],
          ...,
          [2813,   36, 2365,  ..., 1409,  582, 1631],
          [28

In [21]:
inputs['data']['sub_idx']

[tensor([[[    0, 39185, 11503,  ..., 21495, 25379, 28748],
          [    1, 16100,  4242,  ..., 26698, 32836, 10442],
          [    2,  7485, 22153,  ...,  9404,  7939, 20519],
          ...,
          [11261, 12148, 32742,  ..., 24465, 12022, 20106],
          [11262, 33881,  5051,  ..., 36976, 33237,  1195],
          [11263, 33339, 19040,  ..., 30959,  1238, 43442]]]),
 tensor([[[    0,  2177,  1145,  ..., 10323,  3308,  3018],
          [    1,  4242,  7315,  ...,  8062,  6280,   853],
          [    2,  7485,  3251,  ...,  6379,  8162, 10549],
          ...,
          [ 2813,    36, 10383,  ...,  4349,  6046,  7945],
          [ 2814, 10350,  7143,  ...,   734,   249,  3845],
          [ 2815,  7138,  6995,  ...,  2136,   290,  1864]]]),
 tensor([[[   0, 2177, 1145,  ..., 2729,  455,   30],
          [   1, 1247,  506,  ..., 1033, 1347, 2466],
          [   2,  625,  590,  ..., 2585,   69, 1236],
          ...,
          [ 701, 1478, 2324,  ..., 2681, 2512, 1905],
          [ 7

In [28]:
inputs["data"]["features"]

tensor([[[ 4.1933,  5.1708,  0.0477],
         [ 0.4289, -8.6119, -1.6394],
         [ 4.5023,  4.2065, -0.9691],
         ...,
         [-0.1998,  2.4574, -1.3191],
         [ 3.1986, -2.1040, -1.6141],
         [12.2412, -1.2835, -0.9051]]])